In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from tfmap import Atlus
import numpy as np
import polars as pl
import polars.selectors as cs
import matplotlib.pyplot as plt
from typing import Optional
from pybaselines import Baseline
import seaborn as sns
import umap
from sklearn.cluster import KMeans
import scipy.signal as signal
from itertools import pairwise

In [ ]:
np.random.seed(717)

In [ ]:
def atlus_to_df(map_obj: Atlus, cell_type: str) -> pl.DataFrame:
    wn = np.linspace(650, 4000, 3475)
    pixel_idx, pixel_pos = list(zip(*map_obj.pixels.items()))
    pixel_x, pixel_y = list(zip(*pixel_pos))
    pixel_df = pl.DataFrame(dict(idx=pixel_idx, pixel_x=pixel_x, pixel_y=pixel_y))

    spectra_idx, spectra = list(zip(*map_obj.spectra_dict.items()))
    spectra_df = pl.DataFrame(np.array(spectra))
    spectra_df.columns = [f"wavenumber_{x:.2f}" for x in wn]
    spectra_df = spectra_df.with_columns(pl.Series(name="idx", values=spectra_idx))

    return (
        pixel_df.join(spectra_df, on="idx")
        .drop("idx")
        .with_columns(cell_type=pl.lit(cell_type))
    )


def baseline_correction_df(df, plot_res=False):
    clipped_spectra_cols = [
        col
        for col in df.select(cs.contains("wavenumber")).columns
        if 900 <= float(col.split("_")[-1]) <= 3600
    ]
    wavenumbers = [float(col.split("_")[-1]) for col in clipped_spectra_cols]

    baseline_fitter = Baseline(x_data=wavenumbers)
    acc = []
    for example_spectra in df.select(clipped_spectra_cols).to_numpy():
        example_spectra = 2 - np.log10(example_spectra)
        res = baseline_fitter.rubberband(example_spectra, segments=[900, 1900, 2250])[0]
        if plot_res:
            plt.plot(example_spectra - res)
        acc.append(example_spectra - res)
    return pl.concat(
        [
            pl.DataFrame(np.array(acc), schema=clipped_spectra_cols),
            df.select(~cs.contains("wavenumber")),
        ],
        how="horizontal",
    )


def umap_to_kmean(
    df: pl.DataFrame,
    **ftir_params,
):
    n_clusters = ftir_params.get("n_clusters", 2)
    n_neighbors = ftir_params.get("n_neighbors", 15)
    masks = ftir_params.get("masks", None)
    flip = ftir_params.get("flip", False)

    if flip and (n_clusters != 2):
        raise ValueError("flip should only be true if n_clusters == 2")

    if masks is None:
        masks = [(0.0, 5000.0)]

    cols = [
        col
        for col in df.select(cs.contains("wavenumber")).columns
        if any(low <= float(col.split("_")[-1]) <= high for low, high in masks)
    ]
    X = df.select(cols).to_numpy()
    components = umap.UMAP(
        densmap=True, n_neighbors=n_neighbors, random_state=717
    ).fit_transform(X)
    split = KMeans(n_clusters=n_clusters).fit_predict(components)
    if flip:
        split = 1 - split
    return df.with_columns(pc1=components[:, 0], pc2=components[:, 1], split=split)


def ftir_pipeline(
    spec_df: pl.DataFrame, shift_x: int = 3, shift_y: int = -15, **ftir_params
) -> pl.DataFrame:
    corr_df = baseline_correction_df(spec_df, False)
    if ftir_params.get("apply_savgol", False):
        corr_df = savgol(corr_df)
    ukmap_df = umap_to_kmean(corr_df, **ftir_params)
    return ukmap_df.with_columns(
        pl.col("pixel_x").add(shift_x), pl.col("pixel_y").add(shift_y)
    )


def plot_corrected(atlus, df, **ftir_params):
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    atlus.plot_rgb_image(ax=ax)
    pipeline_df = ftir_pipeline(df, **ftir_params)
    sns.scatterplot(
        data=pipeline_df,
        x="pixel_x",
        y="pixel_y",
        hue="split",
        marker="s",
        linewidth=0,
        s=5,
        alpha=0.7,
        ax=ax,
    )
    return pipeline_df, (fig, ax)


def savgol(df: pl.DataFrame, window_length: Optional[int] = None) -> pl.DataFrame:
    spectra = df.select(cs.contains("wavenumber"))
    wavenumbers = [float(col.split("_")[-1]) for col in spectra.columns]
    delta = np.mean([y - x for x, y in pairwise(wavenumbers)])

    if window_length is None:
        window_length = 28

    return pl.concat(
        [
            pl.DataFrame(
                signal.savgol_filter(
                    spectra.to_numpy(),
                    window_length=window_length,
                    polyorder=3,
                    deriv=2,
                    delta=delta,
                ),
                schema=spectra.columns,
            ),
            df.select(~cs.contains("wavenumber")),
        ],
        how="horizontal",
    )

In [ ]:
def load_atlus_frame(filepath: str, sample: str):
    atlus = Atlus.from_map_filepath(filepath)
    df = atlus_to_df(atlus, sample)
    return atlus, df


def load_atlus_frame_multi(
    filepaths: list[str], sample: str, ftir_params: dict
) -> tuple[dict[int, Atlus], pl.DataFrame]:
    acc = []
    lta_dict = dict()
    for idx, filepath in enumerate(filepaths):
        atlus, df = load_atlus_frame(filepath, sample)
        df = df.with_columns(file_idx=idx, filepath=pl.lit(filepath))
        corr_df = ftir_pipeline(df, **ftir_params)
        lta_dict[idx] = atlus
        acc.append(corr_df)
    return lta_dict, pl.concat(acc)

In [ ]:
from typing import Union


def plot_spectra(
    df: pl.DataFrame,
    xlim: Optional[Union[tuple, list]] = None,
    plot_params: Optional[dict] = None,
    write_csv: Optional[str] = None,
):
    fig, ax = plt.subplots(1, 2, figsize=(18, 8))

    if xlim is None:
        low = 0.0
        high = 5000.0
        masks = [(low, high)]
    elif isinstance(xlim, tuple):
        masks = [xlim]
    elif isinstance(xlim, list) and len(xlim) and isinstance(xlim[0], tuple):
        masks = xlim
    else:
        raise ValueError("xlim invalid argument, must be tuple or list of tuples")

    filtered_wavenumbers = [
        col
        for col in df.select(cs.contains("wavenumber")).columns
        if any([low <= float(col.split("_")[1]) <= high for low, high in masks])
    ]
    components = umap.UMAP(densmap=True, n_neighbors=30, min_dist=0.5).fit_transform(
        df.select(filtered_wavenumbers).to_numpy()
    )
    umap_df = df.with_columns(umap_1=components[:, 0], umap_2=components[:, 1])

    if write_csv is not None:
        df.select(
            pl.col(filtered_wavenumbers), ~cs.contains("wavenumber")
        ).with_columns(umap_1=components[:, 0], umap_2=components[:, 1]).write_csv(
            write_csv
        )

    plot_df = (
        df.unpivot(
            cs.contains("wavenumber"),
            index=~cs.contains("wavenumber"),
            variable_name="wavelength",
            value_name="absorbance",
        )
        .with_columns(pl.col("wavelength").str.split("_").list.last().cast(pl.Float64))
        .filter(
            pl.col("wavelength").map_elements(
                lambda wl: any(low <= wl <= high for low, high in masks)
            )
        )
    )

    if plot_params is None:
        plot_params = dict()

    sns.scatterplot(
        data=umap_df,
        x="umap_1",
        y="umap_2",
        hue="cell_type",
        s=3,
        alpha=0.8,
        ax=ax[1],
        **plot_params,
    )
    sns.lineplot(
        data=plot_df,
        x="wavelength",
        y="absorbance",
        hue="cell_type",
        ax=ax[0],
        **plot_params,
    )
    ax[0].xaxis.set_inverted(True)
    return umap_df

In [ ]:
multi_cell_spec_include_df = pl.read_csv("../extra/multi_cell_spec_include_df.csv")

In [ ]:
def select_spectra(df, low, high):
    cols = [
        col
        for col in df.select(cs.contains("wavenumber")).columns
        if low <= float(col.split("_")[1]) <= high
    ]
    return cols


def add_peak_loc_cols(df, name, cols) -> pl.DataFrame:
    peak_name = name + "_peak"
    loc_name = name + "_loc"
    col_name = loc_name + "_col"
    return df.with_columns(
        pl.col(name)
        .map_elements(lambda xs: [abs(x) for x in xs])
        .list.max()
        .alias(peak_name),
        pl.col(name)
        .map_elements(lambda xs: [abs(x) for x in xs])
        .list.arg_max()
        .alias(col_name)
        .map_elements(lambda x: cols[x], return_dtype=pl.String, returns_scalar=True),
    ).with_columns(
        pl.col(col_name).str.split("_").list.last().cast(pl.Float64).alias(loc_name)
    )


def build_amide_df(df) -> pl.DataFrame:
    amide_i_cols = select_spectra(df, low=1600, high=1700)
    amide_ii_cols = select_spectra(df, low=1510, high=1580)
    res = df.with_columns(
        pl.concat_list(amide_i_cols).alias("amide_i"),
        pl.concat_list(amide_ii_cols).alias("amide_ii"),
    )
    res = add_peak_loc_cols(res, "amide_i", amide_i_cols)
    res = add_peak_loc_cols(res, "amide_ii", amide_ii_cols)
    return res


def plot_peak_loc(df, name):
    if name == "amide_i":
        low = 1600
        high = 1700
    elif name == "amide_ii":
        low = 1510
        high = 1580
    else:
        raise ValueError(f"name must be 'amide_i' or 'amide_ii', got: {name}")

    loc = name + "_loc"
    peak = name + "_peak"
    res = build_amide_df(df)
    res_pandas = res.drop("amide_i", "amide_ii").to_pandas()
    fig, ax = plt.subplots(1, 4, figsize=(24, 8))
    loc_mode = res.group_by("cell_type").agg(pl.col(loc).mode())
    print(loc_mode)

    spec_df = (
        df.with_row_index()
        .unpivot(
            cs.contains("wavenumber"),
            index=~cs.contains("wavenumber"),
            variable_name="wavelength",
            value_name="absorbance",
        )
        .with_columns(pl.col("wavelength").str.split("_").list.last().cast(pl.Float64))
        .filter(pl.col("wavelength").gt(low).and_(pl.col("wavelength").lt(high)))
    )
    print(spec_df)

    sns.lineplot(
        data=spec_df,
        x="wavelength",
        y="absorbance",
        hue="cell_type",
        ax=ax[0],
    )

    sns.kdeplot(data=res_pandas, x=peak, hue="cell_type", ax=ax[1])
    sns.kdeplot(data=res_pandas, x=loc, hue="cell_type", ax=ax[2])
    sns.kdeplot(
        data=res_pandas, x=loc, y=peak, hue="cell_type", ax=ax[3], common_grid=True
    )
    ax[0].set_title("Spectra")
    ax[1].set_title("Peak")
    ax[2].set_title("Location")
    ax[3].set_title("Peak + Location")
    fig.suptitle(f"{name}")


def plot_amide_i_vs_ii(df):
    res = build_amide_df(df)
    res = res.with_columns(
        pl.col("amide_i_peak").truediv(pl.col("amide_ii_peak")).alias("i_over_ii")
    )
    res_pandas = res.drop("amide_i", "amide_ii").to_pandas()
    sns.kdeplot(data=res_pandas, x="i_over_ii", hue="cell_type")


In [ ]:
def norm_by_amide_ii(df):
    amide_df = build_amide_df(df)
    print(amide_df)
    amide_df = amide_df.with_columns(
        cs.contains("wavenumber") / pl.col("amide_ii_peak")
    )
    return amide_df

In [ ]:
plot_spectra(
    df=norm_by_amide_ii(multi_cell_spec_include_df).drop("amide_i", "amide_ii"),
    xlim=[(800, 1800), (2800, 3000)],
    plot_params=dict(style="rep"),
)

In [ ]:
plot_spectra(
    df=norm_by_amide_ii(
        multi_cell_spec_include_df.filter(
            ~((pl.col("cell_type") == "ctrl") & (pl.col("rep") == 1))
        )
    ).drop("amide_i", "amide_ii"),
    xlim=[(800, 1800), (2800, 3000)],
    plot_params=dict(style="rep"),
)

In [ ]:
# plot_spectra(
#     norm_by_amide_ii(multi_cell_spec_df).drop(cs.contains("amide")),
#     xlim=(1225, 1300),
# )
plot_spectra(
    df=norm_by_amide_ii(multi_cell_spec_include_df).drop("amide_i", "amide_ii"),
    xlim=(1225, 1300),
    plot_params=dict(style="rep"),
)

In [ ]:
plot_spectra(
    df=savgol(
        norm_by_amide_ii(multi_cell_spec_include_df).drop("amide_i", "amide_ii"),
        window_length=112,
    ),
    xlim=(1225, 1300),
    plot_params=dict(style="rep"),
)

In [ ]:
plot_spectra(
    norm_by_amide_ii(multi_cell_spec_include_df).drop(cs.contains("amide")),
    xlim=(2800, 3000),
)

In [ ]:
plot_spectra(
    savgol(
        norm_by_amide_ii(multi_cell_spec_include_df).drop(cs.contains("amide")),
        window_length=112,
    ),
    xlim=(2800, 3000),
)

In [ ]:
plot_spectra(
    norm_by_amide_ii(multi_cell_spec_include_df).drop(cs.contains("amide")),
    xlim=(800, 1800),
)

In [ ]:
plot_spectra(
    savgol(
        norm_by_amide_ii(multi_cell_spec_include_df).drop(cs.contains("amide")),
        window_length=112,
    ),
    xlim=(800, 1800),
)

In [ ]:
umap_norm_amide_ii_800_1800_multicell_spec_df = plot_spectra(
    norm_by_amide_ii(multi_cell_spec_include_df).drop(cs.contains("amide")),
    xlim=(800, 1800),
    plot_params=dict(style="rep"),
)

In [ ]:
umap_norm_amide_ii_800_1800_multicell_spec_df.write_csv(
    "../extra/umap_norm_amide_ii_800_1800_multicell_spec_df.csv"
)

In [ ]:
plot_spectra(
    savgol(norm_by_amide_ii(multi_cell_spec_include_df), window_length=112).drop(
        "amide_i", "amide_ii"
    ),
    xlim=(1600, 1700),
)
plot_spectra(
    norm_by_amide_ii(multi_cell_spec_include_df).drop("amide_i", "amide_ii"),
    xlim=(1600, 1700),
)
plt.show()
plot_spectra(
    savgol(norm_by_amide_ii(multi_cell_spec_include_df), window_length=112).drop(
        "amide_i", "amide_ii"
    ),
    xlim=(2800, 3000),
)
plt.show()
plot_spectra(
    norm_by_amide_ii(multi_cell_spec_include_df).drop("amide_i", "amide_ii"),
    xlim=(2800, 3000),
)
plt.show()

In [ ]:
umap_savgol_norm_amide_ii_multicell_spec_df = plot_spectra(
    savgol(
        norm_by_amide_ii(multi_cell_spec_include_df).drop(cs.contains("amide")),
        window_length=112,
    ),
)

In [ ]:
umap_savgol_norm_amide_ii_multicell_spec_df = plot_spectra(
    savgol(
        norm_by_amide_ii(multi_cell_spec_include_df).drop(cs.contains("amide")),
        window_length=112,
    ),
    xlim=[(800, 1800), (2800, 3000)],
)

In [ ]:
plot_spectra(
    savgol(norm_by_amide_ii(multi_cell_spec_include_df), window_length=112).drop(
        "amide_i", "amide_ii"
    ),
    xlim=(3000, 3700),
    plot_params=dict(style="rep"),
    write_csv="../extra/multicell-3000to3700umap-savgol.csv",
)

In [ ]:
plot_spectra(
    savgol(norm_by_amide_ii(multi_cell_spec_include_df), window_length=112).drop(
        "amide_i", "amide_ii"
    ),
    xlim=(3300, 3500),
    plot_params=dict(style="rep"),
)

In [ ]:
norm_by_amide_ii(multi_cell_spec_include_df)

In [ ]:
from scipy.stats import ttest_ind
import itertools
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay


def spec_ratio(abs_df: pl.DataFrame, num: str, denom: str) -> pl.DataFrame:
    fig, ax = plt.subplots(1, 4, figsize=(18, 7))
    ratio_col = f"ratio_{num}_{denom}"
    abs_df = abs_df.with_columns(pl.col(num).truediv(pl.col(denom)).alias(ratio_col))
    ratio_by_cell = {
        name: group_df[ratio_col].to_numpy()
        for name, group_df in abs_df.group_by("cell_type")
    }

    for idx, ((a, aarr), (b, barr)) in enumerate(itertools.combinations(ratio_by_cell.items(), 2), start=1):
        mask_a = np.isfinite(aarr)
        mask_b = np.isfinite(barr)
        aarr = aarr[mask_a]
        barr = barr[mask_b]
        t_stat, pvalue = ttest_ind(aarr, barr)
        print(f"t-test: {a}, {b} t-stat: {t_stat:.3f}, pvalue: {pvalue:.3f}")
        estimator = LogisticRegression()

        X = np.array(list(aarr) + list(barr)).reshape(-1, 1)
        y = [a] * aarr.shape[0] + [b] * barr.shape[0]
        estimator.fit(X, y)
        RocCurveDisplay.from_estimator(estimator, X, y, plot_chance_level=True, ax=ax[idx])
        ax[idx].set_title(f"{a} vs. {b}")
        # plt.show()

    sns.violinplot(data=abs_df.to_pandas(), x="cell_type", y=ratio_col, ax=ax[0])
    ax[0].set_ylim(0, 4.0)
    ax[0].set_ylabel(rf"$l_{{{num}}}$/$l_{{{denom}}}$")
    ax[0].set_title(f"Relative Intensity Ratio {num} $cm^{{-1}}$ / {denom} $cm^{{-1}}$")
    plt.show()


for num, denom in [
    (3015.44, 2929.62),
    (2960.48, 2929.62),
    (2929.62, 1072.37),
    (1032.83, 1072.37),
]:
    spec_ratio(
        abs_df=norm_by_amide_ii(multi_cell_spec_include_df),
        num=f"wavenumber_{num}",
        denom=f"wavenumber_{denom}",
    )